In [1]:
import logging

import openeo.processes
from utils import utils

import openeo

logging.basicConfig(level=logging.INFO)

In [2]:
connection = openeo.connect("openeo.dataspace.copernicus.eu")

INFO:openeo.config:Loaded openEO client config from sources: []


In [3]:
connection.authenticate_oidc()

INFO:openeo.rest.connection:Found OIDC providers: ['CDSE']
INFO:openeo.rest.connection:No OIDC provider given, but only one available: 'CDSE'. Using that one.
INFO:openeo.rest.connection:Using default client_id 'sh-b1c3a958-52d4-40fe-a333-153595d1c71e' from OIDC provider 'CDSE' info.
INFO:openeo.rest.connection:Found refresh token: trying refresh token based authentication.
INFO:openeo.rest.auth.oidc:Doing 'refresh_token' token request 'https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token' with post data fields ['grant_type', 'client_id', 'refresh_token'] (client_id 'sh-b1c3a958-52d4-40fe-a333-153595d1c71e')
INFO:openeo.rest.connection:Obtained tokens: ['token_type', 'access_token', 'expires_in', 'id_token', 'refresh_token', 'scope']
INFO:openeo.rest.auth.config:Storing refresh token for issuer 'https://identity.dataspace.copernicus.eu/auth/realms/CDSE' (client 'sh-b1c3a958-52d4-40fe-a333-153595d1c71e')


Authenticated using refresh token.


<Connection to 'https://openeo.dataspace.copernicus.eu/openeo/1.2/' with OidcBearerAuth>

In [4]:
spatial_extent = {
    "west": 30.55,
    "south": 1.07,
    "east": 31.23,
    "north": 1.55,
}

resample_spatial_resolution = 30  # m

temporal_variability_threshold = 0.5  # units: dB
flattening_threshold = 0.12  # units: dimensionless
logistic_sse_threshold = 18.3  # units: dB^2

min_connected_area = 10000  # m^2

In [5]:
# more restrictive (excludes more pixels from being detected as deforestation) than default
temporal_variability_threshold = 0.6
flattening_threshold = 0.14

# Script

In [6]:
# collect outputs as we go, to built a multi-result process graph
process_graph_results = []

## Sentinel 1

In [70]:
# load results from previous batch job
JOB_ID = "j-261119103751464998796dca74b19409"

s1_dB = connection.load_stac_from_job(
    JOB_ID,
    spatial_extent=spatial_extent,
    bands=["sd", "p05", "p95", "min_sse", "min_sse_t"],
)

In [71]:
# fix for nodata tiling artifacts
# https://forum.dataspace.copernicus.eu/t/nodata-tiling-artifacts/5732/6

s1_dB.result_node().update_arguments(featureflags={"retain_nodata_tiles": True})

In [72]:
process_graph_results.append(
    s1_dB.save_result(
        format="NetCDF",
        options={
            "filename_prefix": "0000_s1_dB",
        },
    )
)

In [73]:
# load_stac adds a time dimension 😠
s1_dB = s1_dB.drop_dimension("t")

In [74]:
process_graph_results.append(
    s1_dB.save_result(
        format="NetCDF",
        options={
            "filename_prefix": "0001_s1_dB",
        },
    )
)
process_graph_results.append(
    s1_dB.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0001_s1_dB",
        },
    )
)

# load forest baseline

In [75]:
# load results from previous batch job
JOB_ID = "j-261119103751464998796dca74b19409"

forest_baseline_mask = connection.load_stac_from_job(
    JOB_ID,
    spatial_extent=spatial_extent,
    # note: important to specify temporal_extent here
    # otherwise openEO writes STAC temporal interval [null, null]
    temporal_extent="2020-01-01",
    bands=["B0"],
)

In [77]:
process_graph_results.append(
    forest_baseline_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0010_forest_baseline_mask",
        },
    )
)
process_graph_results.append(
    forest_baseline_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0010_forest_baseline_mask",
        },
    )
)

In [78]:
# load_stac adds a time dimension 😠
forest_baseline_mask = forest_baseline_mask.drop_dimension("t")

In [79]:
process_graph_results.append(
    forest_baseline_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0012_forest_baseline_mask",
        },
    )
)
process_graph_results.append(
    forest_baseline_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0012_forest_baseline_mask",
        },
    )
)

In [80]:
# resample onto S1 grid @ resample_spatial_resolution
forest_baseline_mask = forest_baseline_mask.resample_cube_spatial(
    s1_dB,
    method="near",  # Reference implementation uses rioxarray reproject_match() which has default Resampling.nearest
)

In [81]:
process_graph_results.append(
    forest_baseline_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0015_forest_baseline_mask",
        },
    )
)
process_graph_results.append(
    forest_baseline_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0015_forest_baseline_mask",
        },
    )
)

## temporal variability mask

In [82]:
# mask of interesting pixels

# band math
temporal_variability_mask = s1_dB.band("sd") >= temporal_variability_threshold

In [83]:
process_graph_results.append(
    temporal_variability_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0020_temporal_variability_mask",
        },
    )
)
process_graph_results.append(
    temporal_variability_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0020_temporal_variability_mask",
        },
    )
)

# flattening mask

In [84]:
# ATBD says denominator is abs(gamma_max)
# but `2_deforesation_tracker.ipynb` uses p05
# Dascalu 2023 has abs(gamma_max)


def flattening_reducer(
    bands: openeo.processes.ProcessBuilder,
) -> openeo.processes.ProcessBuilder:
    p05 = bands.array_element(label="p05")
    p95 = bands.array_element(label="p95")
    return (p95 - p05) / p05.absolute()


flattening = s1_dB.reduce_bands(flattening_reducer)

In [85]:
assert flattening._in_bandmath_mode()

In [86]:
process_graph_results.append(
    flattening.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0030_flattening",
        },
    )
)
process_graph_results.append(
    flattening.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0030_flattening",
        },
    )
)

In [87]:
# mask of good pixels where flattening >= flattening_threshold

# band math
flattening_mask = flattening >= flattening_threshold

In [88]:
process_graph_results.append(
    flattening_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0040_flattening_mask",
        },
    )
)
process_graph_results.append(
    flattening_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0040_flattening_mask",
        },
    )
)

# SSE goodness of fit mask

In [89]:
sse_threshold_mask = s1_dB.band("min_sse") <= logistic_sse_threshold

In [90]:
process_graph_results.append(
    sse_threshold_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0050_sse_threshold_mask",
        },
    )
)
process_graph_results.append(
    sse_threshold_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0050_sse_threshold_mask",
        },
    )
)

# Combine all masks

In [91]:
# forest_baseline_mask.band("B0") & temporal_variability_mask & flattening_mask & sse_threshold_mask
# BandMathException: 'Band math' between bands of different data cubes is not supported yet. 🙁

In [92]:
# in order to do band math, we need to stack all of the inputs into a single DataCube
# https://github.com/Open-EO/openeo-python-client/issues/748

b1 = forest_baseline_mask.rename_labels(
    dimension="bands", target=["forest_baseline_mask"]
)
b2 = temporal_variability_mask.add_dimension(
    "bands", label="temporal_variability_mask", type="bands"
)
b3 = flattening_mask.add_dimension("bands", label="flattening_mask", type="bands")
b4 = sse_threshold_mask.add_dimension("bands", label="sse_threshold_mask", type="bands")

combined_masks = b1.merge_cubes(b2).merge_cubes(b3).merge_cubes(b4)

In [93]:
process_graph_results.append(
    combined_masks.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0060_combined_masks",
        },
    )
)
process_graph_results.append(
    combined_masks.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0060_combined_masks",
        },
    )
)
# check dtype of the merge_cubes bands - uint8

In [94]:
# deforestation event = 1
deforestation_event_mask = (
    combined_masks.band("temporal_variability_mask")
    & combined_masks.band("flattening_mask")
    & combined_masks.band("forest_baseline_mask")
    & combined_masks.band("sse_threshold_mask")
)

In [95]:
process_graph_results.append(
    deforestation_event_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0061_deforestation_event_mask",
        },
    )
)
process_graph_results.append(
    deforestation_event_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0061_deforestation_event_mask",
        },
    )
)

In [96]:
inverse_deforestation_event_mask = utils.logical_not(deforestation_event_mask)

In [97]:
process_graph_results.append(
    inverse_deforestation_event_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0062_inverse_deforestation_event_mask",
        },
    )
)
process_graph_results.append(
    inverse_deforestation_event_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0062_inverse_deforestation_event_mask",
        },
    )
)

# Apply deforestation event mask to `min_sse_t`

In [98]:
min_sse_t_masked = s1_dB.band("min_sse_t").mask(inverse_deforestation_event_mask)

In [99]:
process_graph_results.append(
    min_sse_t_masked.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0070_min_sse_t_masked",
        },
    )
)
process_graph_results.append(
    min_sse_t_masked.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0070_min_sse_t_masked",
        },
    )
)

## mask based on connectivity

of the natural forest remaining, are regions of forest too small to meet the minimum connected area threshold?

In [100]:
remaining_forest_mask = (
    combined_masks.band("forest_baseline_mask") & inverse_deforestation_event_mask
)

In [101]:
process_graph_results.append(
    remaining_forest_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0080_remaining_forest_mask",
        },
    )
)
process_graph_results.append(
    remaining_forest_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0080_remaining_forest_mask",
        },
    )
)

In [102]:
connectivity_udf = openeo.UDF.from_file(
    "../udf/connectivity_mask.py",
    runtime="Python",
    version="3.11",
    context={
        "spatial_resolution": resample_spatial_resolution,
        "min_connected_area": min_connected_area,
    },
)

In [103]:
# mask where 1 = small region to be excluded
small_region_mask = remaining_forest_mask.apply_neighborhood(
    connectivity_udf,
    size=[
        {"dimension": "x", "value": 256, "unit": "px"},
        {"dimension": "y", "value": 256, "unit": "px"},
    ],
    # overlap needs to be big enough the reasonably allow for min_pixels
    overlap=[
        {"dimension": "x", "value": 32, "unit": "px"},
        {"dimension": "y", "value": 32, "unit": "px"},
    ],
)

In [104]:
process_graph_results.append(
    small_region_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0090_small_region_mask",
        },
    )
)

In [105]:
# apply_neighborhood UDF seems to return float32, even if it's a mask
# data types: https://github.com/locationtech/geotrellis/blob/master/raster/src/main/scala/geotrellis/raster/CellType.scala
small_region_mask = small_region_mask.convert_data_type("bool")

In [106]:
process_graph_results.append(
    small_region_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0091_small_region_mask",
        },
    )
)

In [107]:
# stack min_sse_t_masked + small_region_mask into a single datacube
# in preparation for the nearest neighbour fill UDF

_data = min_sse_t_masked.add_dimension("bands", label="data", type="bands")
_mask = small_region_mask.add_dimension("bands", label="mask", type="bands")
combined = _data.merge_cubes(_mask)

In [108]:
process_graph_results.append(
    combined.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0100_combined",
        },
    )
)
process_graph_results.append(
    combined.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0100_combined",
        },
    )
)

In [109]:
nearest_neighbour_fill_udf = openeo.UDF.from_file(
    "../udf/nearest_neighbour_fill.py",
    runtime="Python",
    version="3.11",
)

In [110]:
min_sse_t = combined.apply_neighborhood(
    nearest_neighbour_fill_udf,
    size=[
        {"dimension": "x", "value": 256, "unit": "px"},
        {"dimension": "y", "value": 256, "unit": "px"},
    ],
    overlap=[
        {"dimension": "x", "value": 32, "unit": "px"},
        {"dimension": "y", "value": 32, "unit": "px"},
    ],
)

In [111]:
process_graph_results.append(
    min_sse_t.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0110_min_sse_t",
        },
    )
)
process_graph_results.append(
    min_sse_t.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0110_min_sse_t",
        },
    )
)

Include forest baseline mask as a band in the final output

In [112]:
b1 = min_sse_t.rename_labels(dimension="bands", target=["year_of_deforestation"])
b2 = forest_baseline_mask.rename_labels(dimension="bands", target=["forest_baseline"])

output = b1.merge_cubes(b2)

In [113]:
process_graph_results.append(
    output.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0120_output",
        },
    )
)
process_graph_results.append(
    output.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0120_output",
        },
    )
)

# Run batch job

In [114]:
multi_result = openeo.MultiResult(process_graph_results)

In [115]:
job = multi_result.create_job()
job.start_and_wait()

0:00:00 Job 'j-2611201326174acc936c35a3be8adae7': send 'start'
0:00:03 Job 'j-2611201326174acc936c35a3be8adae7': created (progress 0%)
0:00:09 Job 'j-2611201326174acc936c35a3be8adae7': queued (progress 0%)
0:00:15 Job 'j-2611201326174acc936c35a3be8adae7': queued (progress 0%)
0:00:24 Job 'j-2611201326174acc936c35a3be8adae7': queued (progress 0%)
0:00:34 Job 'j-2611201326174acc936c35a3be8adae7': queued (progress 0%)
0:00:46 Job 'j-2611201326174acc936c35a3be8adae7': queued (progress 0%)
0:01:02 Job 'j-2611201326174acc936c35a3be8adae7': queued (progress 0%)
0:01:21 Job 'j-2611201326174acc936c35a3be8adae7': error (progress N/A)
Your batch job 'j-2611201326174acc936c35a3be8adae7' failed. Error logs:
[{'id': '[1795181247228, 261128]', 'time': '2026-11-20T13:27:27.228Z', 'level': 'error', 'message': 'OpenEO batch job failed: LoadStacException(status_code=500, code=\'LoadStacFailure\', message="Error when constructing data cube from load_stac(\'https://openeo.dataspace.copernicus.eu/openeo/1.2

JobFailedException: Batch job 'j-2611201326174acc936c35a3be8adae7' didn't finish successfully. Status: error (after 0:01:22).

In [ ]:
results = job.get_results()

In [ ]:
!mkdir -p output-script/
!rm -r output-script/

In [ ]:
results.download_files("output-script/")

In [ ]:
import json

with open("logs.json", "w") as f:
    json.dump(job.logs(), f, indent=2)